# 1. Feature Store: development

This notebook creates a small SageMaker Feature Store group, ingests records, and demonstrates the online and offline roles. Run it with a dev bucket and an IAM identity that can use SageMaker and S3.

In [ ]:
import time
import boto3
from botocore.exceptions import ClientError

REGION = "us-east-1"
BUCKET = "replace-with-your-dev-bucket"
FEATURE_GROUP = "taxi-trip-features-dev"
sm = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-featurestore-runtime", region_name=REGION)

## Create the feature group

The offline store is useful for historical training data. The online store is enabled here too so the same feature definition can later support low-latency inference.

In [ ]:
try:
    response = sm.create_feature_group(
        FeatureGroupName=FEATURE_GROUP,
        RecordIdentifierFeatureName="trip_id",
        EventTimeFeatureName="event_time",
        FeatureDefinitions=[
            {"FeatureName": "trip_id", "FeatureType": "String"},
            {"FeatureName": "event_time", "FeatureType": "String"},
            {"FeatureName": "trip_distance", "FeatureType": "Fractional"},
            {"FeatureName": "passenger_count", "FeatureType": "Integral"},
        ],
        OnlineStoreConfig={"EnableOnlineStore": True},
        OfflineStoreConfig={"S3StorageConfig": {"S3Uri": f"s3://{BUCKET}/feature-store/offline/"}},
    )
    print(response["FeatureGroupArn"])
except ClientError as error:
    if error.response["Error"]["Code"] == "ResourceInUse":
        print("Feature group already exists")
    else:
        raise

In [ ]:
sm.get_waiter("feature_group_exists").wait(FeatureGroupName=FEATURE_GROUP)
print("Feature group is available")

## Ingest records and read them back

Event time is part of every record. In a real pipeline, feature engineering would happen in a Processing job or pipeline step before this ingestion call.

In [ ]:
now = str(time.time())
records = [
    [{"FeatureName": "trip_id", "ValueAsString": "trip-001"}, {"FeatureName": "event_time", "ValueAsString": now}, {"FeatureName": "trip_distance", "ValueAsString": "4.2"}, {"FeatureName": "passenger_count", "ValueAsString": "2"}],
    [{"FeatureName": "trip_id", "ValueAsString": "trip-002"}, {"FeatureName": "event_time", "ValueAsString": now}, {"FeatureName": "trip_distance", "ValueAsString": "1.8"}, {"FeatureName": "passenger_count", "ValueAsString": "1"}],
]
for record in records:
    runtime.put_record(FeatureGroupName=FEATURE_GROUP, Record=record)
print("Inserted records")
print(runtime.get_record(FeatureGroupName=FEATURE_GROUP, RecordIdentifierValueAsString="trip-001"))

## What to explore next

Query the offline store through Athena after records have landed in S3. Compare the point-in-time training dataset with the online record used for inference. This is where feature freshness and training-serving skew become practical concerns.